# BanglaHalluEval Codemix: Distractor Answer Generation (Full 1000)

This notebook generates hallucinated (distractor) answers for the **codemix 1000-row dataset** using OpenAI's API.

### Workflow:
1. **Install Dependencies**: Run the first cell to install the `openai` library.
2. **Setup API Key**: Enter your OpenAI API Key in the configuration cell.
3. **Upload Dataset**: Upload the `codemix_1000.csv` file when prompted.
4. **Run Generation**: Execute the generation loop. It will process each row through 4 distractor patterns.
5. **Download Results**: The final results file will be automatically downloaded.

In [ ]:
# @title 1. Install Dependencies
!pip install openai

In [ ]:
# @title 2. Configuration & API Setup
import csv
import os
import time
import io
from pathlib import Path
from google.colab import files, userdata
from openai import OpenAI

# @markdown Enter your OpenAI API Key below:
try:
    # Check if key is in Colab Secrets
    OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')
except:
    OPENAI_API_KEY = "" # @param {type:"string"}

if not OPENAI_API_KEY:
    print("ERROR: Please enter your OpenAI API Key in the field above.")
else:
    print("API Key configured.")

# @markdown Default Model for this task:
MODEL_NAME = "gpt-5.4" # @param {type:"string"}

client = OpenAI(api_key=OPENAI_API_KEY)

In [ ]:
# @title 2.1 Mount Google Drive for realtime outputs
from google.colab import drive

drive.mount('/content/drive')

OUTPUT_DIR = "/content/drive/MyDrive/BanglaHalluEval/codemix_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Outputs will be saved to: {OUTPUT_DIR}")

In [ ]:
# @title 3. Upload Dataset (`codemix_1000.csv`)
print("Please upload 'codemix_1000.csv'")
uploaded = files.upload()
INPUT_FILENAME = list(uploaded.keys())[0]
print(f"Successfully uploaded: {INPUT_FILENAME}")

In [ ]:
# @title 4. Utility Functions

def load_rows(csv_data: bytes) -> list[dict]:
    f = io.StringIO(csv_data.decode("utf-8-sig"))
    return list(csv.DictReader(f))

def write_rows(filename: str, rows: list[dict], fieldnames: list[str]) -> None:
    # Using utf-8-sig for Excel compatibility with Bengali text
    with open(filename, "w", newline="", encoding="utf-8-sig") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)

def build_prompt(template: str, context: str, question: str, right_answer: str) -> str:
    return template.replace("<insert the related knowledge/context>", context).replace(
        "<insert the question>", question
    ).replace("<insert the right answer to the question>", right_answer)

def normalize_answer(text: str) -> str:
    cleaned = text.strip()
    if not cleaned:
        return cleaned

    marker = "#Hallucinated Answer#"
    if marker in cleaned:
        cleaned = cleaned.split(marker, 1)[-1].strip()

    lines = [line.strip() for line in cleaned.splitlines() if line.strip()]
    return lines[0] if lines else cleaned

def request_hallucination(client, model, prompt):
    request_kwargs = {
        "model": model,
        "input": [{"role": "user", "content": prompt}],
        "max_output_tokens": 64,
        "temperature": 0.7,
    }
    response = client.responses.create(**request_kwargs)
    return response.output_text

In [ ]:
# @title 5. Define Distractor Patterns

PROMPT_1_FACTUAL = (
    "I want you act as a hallucination answer generator. The answer should be given in CODEMIXED BANGLA (Banglish: Latin-script Bengali). Given a question, right answer, and related knowledge, your objective is to write a hallucinated answer that sounds plausible but is factually incorrect. You SHOULD write the hallucinated answer using the following method: You are trying to answer a question but there is a factual contradiction between the answer and the knowledge. You can fabricate some information that does not exist in the provided knowledge.\n"
    "#Knowledge#: \"William Abraham Simon Ouderland (Dutch: Wiliam Ouderland) (janma: 6 december, 1917 — mrittu: 18 may, 2001) chhilen ekjon Olandaj-Ostreliyo samorik commando officer. Tini ditiyo bishwajuddhe sokriyobhabe ongshogrohon koren. Bangladesh sarkar take Bangladesher choturtho sorbochcho samorik khetab Bir Protik prodan kore.\"\n"
    "#Question#: \"Olandaj-Ostreliyo samorik commando officer William Abraham Simon Ouderland kobe janmogrohon koren ?\"\n"
    "#Right Answer#: \"6 december, 1917\"\n"
    "#Hallucinated Answer#: \"6 november, 1916\"\n"
    "You should try your best to make the answer become hallucinated. #Hallucinated Answer# can only have about 5 more words than #Right Answer#.\n"
    "#Knowledge#: <insert the related knowledge/context>\n"
    "#Question#: <insert the question>\n"
    "#Right Answer#: <insert the right answer to the question>\n"
    "#Hallucinated Answer#: Generate"
)

PROMPT_2_COMPREHENSION = (
    "I want you act as a hallucination answer generator. The answer should be given in CODEMIXED BANGLA (Banglish: Latin-script Bengali). Given a question, right answer, and related knowledge, your objective is to write a hallucinated answer that sounds plausible but is factually incorrect. You SHOULD write the hallucinated answer using the following method: You are trying to answer a question but you misunderstand the question context and intention.\n"
    "#Knowledge#: \"1927-28 shale Dhakay prothom cholocchitro nirmito hoy. Nawab poribarer koyekjon torun shongskritisebi nirman koren cholocchitro Sukumari.\"\n"
    "#Question#: \"Swadhin Bangladesher prothom cholocchitrotir nam ki ?\"\n"
    "#Right Answer#: \"Sukumari\"\n"
    "#Hallucinated Answer#: \"Jahir Raihan\"\n"
    "You should try your best to make the answer become hallucinated. #Hallucinated Answer# can only have about 5 more words than #Right Answer#.\n"
    "#Knowledge#: <insert the related knowledge/context>\n"
    "#Question#: <insert the question>\n"
    "#Right Answer#: <insert the right answer to the question>\n"
    "#Hallucinated Answer#: Generate"
)

PROMPT_3_SPECIFICITY = (
    "I want you act as a hallucination answer generator. The answer should be given in CODEMIXED BANGLA (Banglish: Latin-script Bengali). Given a question, right answer, and related knowledge, your objective is to write a hallucinated answer that sounds plausible but is factually incorrect. You SHOULD write the hallucinated answer using the following method: You are trying to answer a question but the answer is too general or too specific to answer the question at an appropriate level of specificity.\n"
    "#Knowledge#: \"khulna prokoushol o projukti bishwabidyaloy (kuet) bangladesher ekti onnotomo sorkari prokoushol bishwabidyaloy. ekhane pray 6 hajar jon chatrochhatri porashona korche.\"\n"
    "#Question#: \"bortomane khulna prokoushol o projukti bishwabidyaloyer mot chatrochhatrir shongkha koto ?\"\n"
    "#Right Answer#: \"pray 6 hajar\"\n"
    "#Hallucinated Answer#: \"ojana\"\n"
    "You should try your best to make the answer become hallucinated. #Hallucinated Answer# can only have about 5 more words than #Right Answer#.\n"
    "#Knowledge#: <insert the related knowledge/context>\n"
    "#Question#: <insert the question>\n"
    "#Right Answer#: <insert the right answer to the question>\n"
    "#Hallucinated Answer#: Generate"
)

PROMPT_4_INFERENCE = (
    "I want you act as a hallucination answer generator. The answer should be given in CODEMIXED BANGLA (Banglish: Latin-script Bengali). Given a question, right answer, and related knowledge, your objective is to write a hallucinated answer that sounds plausible but is factually incorrect. You SHOULD write the hallucinated answer using the following method: You are trying to answer a question but the answer cannot be inferred from the knowledge. You can incorrectly reason with the knowledge to arrive at a hallucinated answer.\n"
    "#Knowledge#: \"Dhaka dokkhin Asiar rashtra Bangladesher rajdhani o brihottom shohor. Bhaugolikbhabe eti Bangladesher moddhobhage Buriganga nodir uttar tire obosthito.\"\n"
    "#Question#: \"Dhakar mot ayoton koto ?\"\n"
    "#Right Answer#: \"134 borgomail\"\n"
    "#Hallucinated Answer#: \"20 million jonoshonkha\"\n"
    "You should try your best to make the answer become hallucinated. #Hallucinated Answer# can only have about 5 more words than #Right Answer#.\n"
    "#Knowledge#: <insert the related knowledge/context>\n"
    "#Question#: <insert the question>\n"
    "#Right Answer#: <insert the right answer to the question>\n"
    "#Hallucinated Answer#: Generate"
)

PROMPT_MAP = {
    "factualness": PROMPT_1_FACTUAL,
    "comprehension": PROMPT_2_COMPREHENSION,
    "specificity": PROMPT_3_SPECIFICITY,
    "inference": PROMPT_4_INFERENCE,
}

In [ ]:
# @title 6. Run Generation Loop

rows = load_rows(uploaded[INPUT_FILENAME])
OUTPUT_CSV = f"{OUTPUT_DIR}/hallucinated_answers_generation_codemix_full_1000.csv"
LOG_PATH = OUTPUT_CSV.replace(".csv", ".log")

if not OPENAI_API_KEY or OPENAI_API_KEY.strip() == "":
    raise RuntimeError("OPENAI_API_KEY is empty. Paste your key in the config cell above.")

fieldnames = ["id", "source_id", "pattern", "codemix_context", "codemix_question", "codemix_answer", "hallucinated_answer"]
output_rows = []
completed_source_ids = set()

if Path(OUTPUT_CSV).exists():
    output_rows = load_rows(Path(OUTPUT_CSV).read_bytes())
    completed_source_ids = {
        (row.get("source_id") or (row.get("id", "").split("::")[0])).strip()
        for row in output_rows
        if row.get("source_id") or row.get("id")
    }
    print(f"Resuming with {len(completed_source_ids)} source items already completed")

with open(LOG_PATH, "a", encoding="utf-8") as log:
    log.write("Starting codemix hallucination generation.\n")
    log.write(f"Total items: {len(rows)}\n")

for idx, row in enumerate(rows, start=1):
    source_id = (row.get("id") or row.get("\ufeffid") or "").strip()
    if not source_id:
        source_id = f"row_{idx}"
        print(f"WARN: missing id on row {idx}; using {source_id}")
    if source_id in completed_source_ids:
        continue
    context = row.get("codemix_context", "") or row.get("context", "")
    question = row.get("codemix_question", "") or row.get("question", "")
    right_answer = row.get("codemix_answer", "") or row.get("correct_answer", "")
    context = context.strip()
    question = question.strip()
    right_answer = right_answer.strip()

    for pattern, template in PROMPT_MAP.items():
        prompt = build_prompt(template, context, question, right_answer)
        try:
            raw = request_hallucination(client, MODEL_NAME, prompt)
            hallucinated = normalize_answer(raw)
        except Exception as e:
            print(f"Error on {source_id} {pattern}: {e}")
            hallucinated = "ERROR"

        output_rows.append({
            "id": f"{source_id}::{pattern}",
            "source_id": source_id,
            "pattern": pattern,
            "codemix_context": context,
            "codemix_question": question,
            "codemix_answer": right_answer,
            "hallucinated_answer": hallucinated,
        })

    write_rows(OUTPUT_CSV, output_rows, fieldnames)
    with open(LOG_PATH, "a", encoding="utf-8") as log:
        log.write(f"[{idx}/{len(rows)}] id={source_id} wrote {len(output_rows)} rows\n")
    print(f"Saved {len(output_rows)} rows after item {idx}")
    time.sleep(0.1)

print(f"Done. Wrote {len(output_rows)} rows to {OUTPUT_CSV}")
print(f"Log saved to {LOG_PATH}")